# ntl-refll

Per-token likelihood of every train passage under Qwen3-4B-Base and Qwen3-8B-Base (first 160 tokens): a neutral reference for the GLM-5.3 likelihood features of exp050.

In [ ]:
%%writefile /tmp/refll.py
import glob, json, os, sys, time
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

CUT = 160


def score_texts(path, texts, batch=8):
    """Per text an (n, 4) float32 array: log p(true token), entropy, E_p[log p ^ 2], rank (tokens above the true one),
    for the first CUT tokens after an <|endoftext|> context token."""
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map="auto").eval()
    bos = tok.convert_tokens_to_ids("<|endoftext|>")
    enc = [[bos] + tok(t, add_special_tokens=False).input_ids[:CUT] for t in texts]
    dev = model.get_input_embeddings().weight.device
    out = [None] * len(enc)
    order = np.argsort([len(e) for e in enc], kind="stable")
    with torch.no_grad():
        for b in range(0, len(order), batch):
            idx = order[b:b + batch]
            L = max(len(enc[i]) for i in idx)
            x = torch.full((len(idx), L), bos, dtype=torch.long)
            att = torch.zeros((len(idx), L), dtype=torch.long)
            for r, i in enumerate(idx):
                x[r, :len(enc[i])] = torch.tensor(enc[i])
                att[r, :len(enc[i])] = 1
            logits = model(input_ids=x.to(dev), attention_mask=att.to(dev)).logits[:, :-1].float()
            lpa = torch.log_softmax(logits, -1)
            del logits
            lp = lpa.gather(-1, x[:, 1:].to(lpa.device)[..., None])[..., 0]
            p = lpa.exp()
            h = -(p * lpa).sum(-1)
            m2 = (p * lpa * lpa).sum(-1)
            rank = (lpa > lp[..., None]).sum(-1).float()
            del p, lpa
            s = torch.stack([lp, h, m2, rank], -1).cpu().numpy().astype(np.float32)
            for r, i in enumerate(idx):
                out[i] = s[r, :len(enc[i]) - 1]
    del model
    torch.cuda.empty_cache()
    return out


In [ ]:
import glob, json, os, time
import numpy as np, pandas as pd
exec(open('/tmp/refll.py').read())
TRAIN = next(p for p in glob.glob('/kaggle/input/**/train.csv', recursive=True) if 'name-that-llm' in p)
df = pd.read_csv(TRAIN)
ids, texts = df.id.astype(str).tolist(), df.text.fillna('').astype(str).tolist()
os.makedirs('/kaggle/working/out', exist_ok=True)
for tag, size in (('q4b', '4b-base'), ('q8b', '8b-base')):
    path = os.path.dirname(sorted(glob.glob(f'/kaggle/input/**/{size}/**/config.json', recursive=True))[0])
    t = time.time()
    sc = score_texts(path, texts)
    off = np.concatenate([[0], np.cumsum([len(s) for s in sc])])
    allsc = np.concatenate(sc)
    bad = int((~np.isfinite(allsc)).any(1).sum())
    np.savez(f'/kaggle/working/out/{tag}_scores.npz', ids=np.array(ids, dtype='U64'), offsets=off, scores=allsc)
    print(f'{tag}: {path}, {len(texts)} texts, {len(allsc)} tokens, non-finite rows {bad}, {time.time() - t:.0f} s', flush=True)
